In [ ]:
#!/usr/bin/env python3"""Run all valid combinations on DiabetesRecord dataset."""import itertoolsimport sysimport ossys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))import pandas as pdimport numpy as npimport kagglehubfrom core.GenericDataPipeline import GenericDataPipelinefrom core.RunData import RunPipelinefrom sklearn.metrics import roc_auc_scorefrom sklearn.model_selection import train_test_splitfrom core.seed_utils import SEED, set_all_seedspd.set_option('future.infer_string', False)set_all_seeds()# --- Config ---N_TRIALS = int(sys.argv[1]) if len(sys.argv) > 1 else 10NULL_THRESHOLD = 0.05  # Features with >5% nulls are candidates for extendedMIN_GROUP_PCT = 0.07   # Min 7% of total population per test grouppipeline = GenericDataPipeline()# --- Load data ---path = kagglehub.dataset_download("brandao/diabetes")csv_path = os.path.join(path, "diabetic_data.csv")df = pd.read_csv(csv_path)# Target: readmitted (NO=0, <30 or >30 = 1)df['readmitted'] = df['readmitted'].replace({'NO': 0, '<30': 1, '>30': 1}).astype(int)# Drop columns as in notebookcolumns_to_drop = ['encounter_id', 'patient_nbr', 'number_inpatient', 'number_emergency', 'discharge_disposition_id']df = df.drop(columns=columns_to_drop)df = pipeline.preprocessing(df)label = "readmitted"print(f"Dataset shape: {df.shape}")print(f"N_TRIALS per model: {N_TRIALS}")print(f"Target distribution:\n{df[label].value_counts()}")# --- EML Feature Comparison ---print(f"\n{'='*120}")print("EML FEATURE COMPARISON")print(f"{'='*120}")from EML.eml_feature_comparison import EMLFeatureComparison# Define extended features (fill in the list with feature names)extended_features = []  # TODO: Add extended feature names hereprint(f"Extended features: {extended_features}")print(f"Running EML comparison...")# Initialize and run EML comparisoneml_comparison = EMLFeatureComparison(    df=df.copy(),    features_extended=extended_features,    target_col=label,    base_learners=None,    meta_learner=None,    cv=5,    use_proba=True,            )# Run the comparisonresult = eml_comparison.run()print(f"\n{'='*120}")print("EML SUMMARY")print(f"{'='*120}")print(f"AUC (Basic features only):  {result['auc_without_extended']:.4f}")print(f"AUC (All features):         {result['auc_with_all']:.4f}")print(f"Difference:                 {result['difference']:+.4f}")print(f"{'='*120}")print("\nEML Analysis Complete!")